### import libs 

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

### paths

In [2]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for path in [start_path, *start_path.parents]:
        if (path / "README.md").exists() and (path / "config.yaml").exists():
            return path

    raise FileNotFoundError("Project root not found.")


PROJECT_ROOT = find_project_root(Path.cwd())

RAW_STOCK_DIR = PROJECT_ROOT / "data" / "raw" / "yfinance" / "stocks"
QUALITY_REPORT_DIR = PROJECT_ROOT / "data" / "interim" / "quality_reports"

LARGE_MOVES_PATH = QUALITY_REPORT_DIR / "large_daily_moves_report.csv"
REVIEW_REPORT_PATH = QUALITY_REPORT_DIR / "large_daily_moves_review_report.csv"

print("Project root:", PROJECT_ROOT)
print("Raw stock dir:", RAW_STOCK_DIR)
print("Large moves path:", LARGE_MOVES_PATH)
print("Review output path:", REVIEW_REPORT_PATH)

Project root: E:\Projects\marketguard-india
Raw stock dir: E:\Projects\marketguard-india\data\raw\yfinance\stocks
Large moves path: E:\Projects\marketguard-india\data\interim\quality_reports\large_daily_moves_report.csv
Review output path: E:\Projects\marketguard-india\data\interim\quality_reports\large_daily_moves_review_report.csv


### Load current large moves report

In [3]:
large_moves = pd.read_csv(LARGE_MOVES_PATH)

large_moves["date"] = pd.to_datetime(large_moves["date"])

print("Large moves shape:", large_moves.shape)
display(large_moves.head())
display(large_moves.tail())

Large moves shape: (51, 14)


,date,symbol,yf_ticker,company_name,industry,open,high,low,close,adj_close,volume,dividends,stock_splits,return_1d
0,2010-05-17,ABB,ABB.NS,ABB India Ltd.,Capital Goods,621.883179,793.468506,621.883179,752.660278,705.385620,3996030,0.0,0.0,0.231141
1,2013-05-17,ABB,ABB.NS,ABB India Ltd.,Capital Goods,499.322296,607.402832,497.733521,598.642029,568.456726,1904344,0.0,0.0,0.209354
2,2023-01-27,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,2406.050049,2448.000000,2014.199951,2014.199951,2014.199951,2529574,0.0,0.0,-0.200000
3,2011-07-29,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,112.197342,112.465576,83.994736,89.742546,75.362930,76281771,0.0,0.0,-0.213619
4,2014-04-10,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,58.849957,73.250153,58.604717,72.016289,66.455940,163256482,0.0,0.0,0.228848


,date,symbol,yf_ticker,company_name,industry,open,high,low,close,adj_close,volume,dividends,stock_splits,return_1d
46,2025-10-14,TMPV,TMPV.NS,Tata Motors Passenger Vehicles Ltd.,Automobile and Auto Components,400.000000,421.549988,376.299988,395.450012,392.196625,52267874,0.0,0.0,-0.401513
47,2017-10-25,UNIONBANK,UNIONBANK.NS,Union Bank of India,Financial Services,144.500000,178.000000,144.500000,176.300003,147.895294,32616021,0.0,0.0,0.341705
48,2012-11-12,UNITDSPR,UNITDSPR.NS,United Spirits Ltd.,Fast Moving Consumer Goods,279.989990,374.920013,275.200012,366.589996,355.862701,126629730,0.0,0.0,0.347262
49,2020-10-12,VEDL,VEDL.NS,Vedanta Ltd.,Metals & Mining,109.699997,109.699997,92.150002,96.949997,42.174454,145043336,0.0,0.0,-0.204350
50,2026-04-30,VEDL,VEDL.NS,Vedanta Ltd.,Metals & Mining,289.500000,292.000000,268.700012,271.549988,271.549988,73870853,0.0,0.0,-0.648979


### Load all raw stock files

In [4]:
stock_files = sorted(RAW_STOCK_DIR.glob("*.csv"))

print("Stock files:", len(stock_files))
display(stock_files[:5])

Stock files: 100


[WindowsPath('E:/Projects/marketguard-india/data/raw/yfinance/stocks/ABB_NS.csv'),
 WindowsPath('E:/Projects/marketguard-india/data/raw/yfinance/stocks/ADANIENSOL_NS.csv'),
 WindowsPath('E:/Projects/marketguard-india/data/raw/yfinance/stocks/ADANIENT_NS.csv'),
 WindowsPath('E:/Projects/marketguard-india/data/raw/yfinance/stocks/ADANIGREEN_NS.csv'),
 WindowsPath('E:/Projects/marketguard-india/data/raw/yfinance/stocks/ADANIPORTS_NS.csv')]

In [5]:
stock_dfs = []

for path in stock_files:
    df = pd.read_csv(path)

    if "date" not in df.columns:
        raise ValueError(f"No date column in {path.name}")

    df["date"] = pd.to_datetime(df["date"])
    df["source_file"] = path.name

    stock_dfs.append(df)

stocks = pd.concat(stock_dfs, ignore_index=True)

print("Stocks shape:", stocks.shape)
display(stocks.head())

Stocks shape: (357370, 16)


,asset_type,symbol,yf_ticker,company_name,industry,date,adj_close,close,dividends,high,low,open,repaired?,stock_splits,volume,source_file
0,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-04,649.419373,694.875061,0.0,703.408936,690.063416,699.051208,False,0.0,204731,ABB_NS.csv
1,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-05,647.765015,693.104736,0.0,700.821533,691.788330,695.419739,False,0.0,175062,ABB_NS.csv
2,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-06,647.977051,693.331665,0.0,700.367615,690.199585,698.733459,False,0.0,222114,ABB_NS.csv
3,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-07,663.758423,710.217834,0.0,717.208374,692.877747,693.604065,False,0.0,438939,ABB_NS.csv
4,stock,ABB,ABB.NS,ABB India Ltd.,Capital Goods,2010-01-08,675.382690,722.655518,0.0,728.919739,711.171082,713.713135,False,0.0,784364,ABB_NS.csv


### Compute raw and adjusted daily returns

In [6]:
required_cols = ["date", "yf_ticker", "close", "adj_close"]

missing = [col for col in required_cols if col not in stocks.columns]
missing

[]

In [7]:
stocks = stocks.sort_values(["yf_ticker", "date"]).copy()

stocks["prev_close"] = stocks.groupby("yf_ticker")["close"].shift(1)
stocks["prev_adj_close"] = stocks.groupby("yf_ticker")["adj_close"].shift(1)

stocks["raw_return_1d"] = stocks["close"] / stocks["prev_close"] - 1
stocks["adj_return_1d"] = stocks["adj_close"] / stocks["prev_adj_close"] - 1

stocks["raw_adj_gap"] = stocks["raw_return_1d"] - stocks["adj_return_1d"]

stocks["abs_raw_return_1d"] = stocks["raw_return_1d"].abs()
stocks["abs_adj_return_1d"] = stocks["adj_return_1d"].abs()
stocks["abs_raw_adj_gap"] = stocks["raw_adj_gap"].abs()

display(
    stocks[
        ["date", "symbol", "yf_ticker", "close", "adj_close", "prev_close", "prev_adj_close",
         "raw_return_1d", "adj_return_1d", "raw_adj_gap"]
    ].head(20)
)

,date,symbol,yf_ticker,close,adj_close,prev_close,prev_adj_close,raw_return_1d,adj_return_1d,raw_adj_gap
0,2010-01-04,ABB,ABB.NS,694.875061,649.419373,NaN,NaN,NaN,NaN,NaN
1,2010-01-05,ABB,ABB.NS,693.104736,647.765015,694.875061,649.419373,-0.002548,-0.002547,-2.462114e-07
2,2010-01-06,ABB,ABB.NS,693.331665,647.977051,693.104736,647.765015,0.000327,0.000327,7.401812e-08
3,2010-01-07,ABB,ABB.NS,710.217834,663.758423,693.331665,647.977051,0.024355,0.024355,2.788889e-07
4,2010-01-08,ABB,ABB.NS,722.655518,675.382690,710.217834,663.758423,0.017512,0.017513,-3.080624e-07
5,2010-01-11,ABB,ABB.NS,745.034241,696.297363,722.655518,675.382690,0.030967,0.030967,2.018387e-07
6,2010-01-12,ABB,ABB.NS,725.606079,678.140198,745.034241,696.297363,-0.026077,-0.026077,-1.304594e-07
7,2010-01-13,ABB,ABB.NS,743.808655,695.151917,725.606079,678.140198,0.025086,0.025086,1.864249e-07
8,2010-01-14,ABB,ABB.NS,775.265930,724.551270,743.808655,695.151917,0.042292,0.042292,1.780237e-07
9,2010-01-15,ABB,ABB.NS,777.399414,726.545471,775.265930,724.551270,0.002752,0.002752,-3.883544e-07


### Create event context counts

This helps identify real sector-wide moves.

In [8]:
large_move_context = stocks[
    stocks["raw_return_1d"].abs() > 0.20
].copy()

same_date_counts = (
    large_move_context
    .groupby("date")
    .size()
    .reset_index(name="same_date_large_move_count")
)

same_date_industry_counts = (
    large_move_context
    .groupby(["date", "industry"])
    .size()
    .reset_index(name="same_date_industry_large_move_count")
)

display(same_date_counts.sort_values("same_date_large_move_count", ascending=False).head(20))
display(same_date_industry_counts.sort_values("same_date_industry_large_move_count", ascending=False).head(20))

,date,same_date_large_move_count
21,2020-03-23,6
16,2017-10-25,5
34,2024-06-04,3
20,2020-03-12,2
0,2010-01-08,1
1,2010-05-17,1
2,2010-05-26,1
6,2013-04-12,1
5,2012-11-12,1
4,2011-07-29,1


,date,industry,same_date_industry_large_move_count
22,2020-03-23,Financial Services,6
16,2017-10-25,Financial Services,5
35,2024-06-04,Financial Services,2
1,2010-05-17,Capital Goods,1
4,2011-07-29,Metals & Mining,1
5,2012-11-12,Fast Moving Consumer Goods,1
2,2010-05-26,Construction Materials,1
3,2011-03-03,Financial Services,1
0,2010-01-08,Fast Moving Consumer Goods,1
8,2014-04-10,Metals & Mining,1


### Merge computed returns into large moves report

In [9]:
review_cols = [
    "date",
    "yf_ticker",
    "prev_close",
    "prev_adj_close",
    "raw_return_1d",
    "adj_return_1d",
    "raw_adj_gap",
    "abs_raw_return_1d",
    "abs_adj_return_1d",
    "abs_raw_adj_gap",
]

optional_cols = [
    "dividends",
    "stock_splits",
]

review_cols = review_cols + [col for col in optional_cols if col in stocks.columns]

returns_for_review = stocks[review_cols].copy()

large_moves_review = large_moves.merge(
    returns_for_review,
    on=["date", "yf_ticker"],
    how="left",
    suffixes=("", "_computed")
)

large_moves_review = large_moves_review.merge(
    same_date_counts,
    on="date",
    how="left"
)

large_moves_review = large_moves_review.merge(
    same_date_industry_counts,
    on=["date", "industry"],
    how="left"
)

large_moves_review["same_date_large_move_count"] = (
    large_moves_review["same_date_large_move_count"].fillna(0).astype(int)
)

large_moves_review["same_date_industry_large_move_count"] = (
    large_moves_review["same_date_industry_large_move_count"].fillna(0).astype(int)
)

print("Review shape:", large_moves_review.shape)
display(large_moves_review.head())

Review shape: (51, 26)


,date,symbol,yf_ticker,company_name,industry,open,high,low,close,adj_close,...,raw_return_1d,adj_return_1d,raw_adj_gap,abs_raw_return_1d,abs_adj_return_1d,abs_raw_adj_gap,dividends_computed,stock_splits_computed,same_date_large_move_count,same_date_industry_large_move_count
0,2010-05-17,ABB,ABB.NS,ABB India Ltd.,Capital Goods,621.883179,793.468506,621.883179,752.660278,705.385620,...,0.231141,0.231140,1.916717e-07,0.231141,0.231140,1.916717e-07,0.0,0.0,1,1
1,2013-05-17,ABB,ABB.NS,ABB India Ltd.,Capital Goods,499.322296,607.402832,497.733521,598.642029,568.456726,...,0.209354,0.209353,4.638409e-07,0.209354,0.209353,4.638409e-07,0.0,0.0,1,1
2,2023-01-27,ADANIENSOL,ADANIENSOL.NS,Adani Energy Solutions Ltd.,Power,2406.050049,2448.000000,2014.199951,2014.199951,2014.199951,...,-0.200000,-0.200000,0.000000e+00,0.200000,0.200000,0.000000e+00,0.0,0.0,1,1
3,2011-07-29,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,112.197342,112.465576,83.994736,89.742546,75.362930,...,-0.213619,-0.213619,1.811191e-07,0.213619,0.213619,1.811191e-07,0.0,0.0,1,1
4,2014-04-10,ADANIENT,ADANIENT.NS,Adani Enterprises Ltd.,Metals & Mining,58.849957,73.250153,58.604717,72.016289,66.455940,...,0.228848,0.228848,-2.220093e-07,0.228848,0.228848,2.220093e-07,0.0,0.0,1,1
